# Audio + Flow Tensor Motion Anomaly Inference

音声モデルは従来通り推論し、動きモデルは学習artifactに保存されたflow系特徴の3x3時間tensorで推論します。音声・動き・同期スコアを統合し、推論結果を表で表示します。


## 処理の流れ

1. 学習artifactを読み込み、保存済み設定で音声特徴とflow tensorを抽出します。
2. 音声・動き・同期スコアをwindow単位で計算します。
3. 動画ごとの最大異常窓と動き寄与を表・CSVにまとめます。


In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        fallback_iterable = [] if iterable is None else iterable
        return fallback_iterable


def _find_import_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'forklift_features').exists():
            return candidate
    raise FileNotFoundError(f'Could not find src/forklift_features from {start}')


PROJECT_ROOT = _find_import_root()
src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from forklift_features import flow_tensor, pipeline as feature_pipeline, scoring
from forklift_features.paths import find_project_root, safe_path_part

flow_tensor = importlib.reload(flow_tensor)
feature_pipeline = importlib.reload(feature_pipeline)
scoring = importlib.reload(scoring)

pd.set_option('display.max_columns', 160)


In [ ]:
PROJECT_ROOT = find_project_root()
MOVIE_PREPROCESS_DIR = PROJECT_ROOT / 'data' / 'movie_preprocess'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'anomaly_feature_poc'
FEATURE_CACHE_DIR = OUTPUT_DIR / 'sample_feature_cache'
MODEL_PATH = OUTPUT_DIR / 'isolation_forest_feature_poc.joblib'
TARGET_OUTPUT_ROOT = OUTPUT_DIR / 'target_inference'
TARGET_COMPARISON_DIR = TARGET_OUTPUT_ROOT / '_comparison'
TARGET_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
TARGET_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Trained model was not found: {MODEL_PATH}')
artifact = joblib.load(MODEL_PATH)
if 'flow_tensor_model' not in artifact or 'score_models' not in artifact or 'audio' not in artifact['score_models']:
    raise ValueError('This inference notebook requires an audio_flow_tensor_sync_v2 artifact. Re-run the training notebook.')
if artifact.get('model_version') != 'audio_flow_tensor_sync_v2':
    raise ValueError('This inference notebook requires an audio_flow_tensor_sync_v2 artifact. Re-run the training notebook.')

settings = artifact.get('settings', {})
AUDIO_SR = int(settings.get('audio_sr', 16000))
N_MELS = int(settings.get('n_mels', 16))
WINDOW_SEC = float(settings.get('window_sec', 0.2))
FLOW_SAMPLE_SEC = float(settings.get('flow_sample_sec', 0.1))
FLOW_GRID = tuple(settings.get('flow_grid', (3, 3)))
USE_FRONT = bool(settings.get('use_front', True))
USE_REAR = bool(settings.get('use_rear', True))
FRAME_RESIZE_WIDTH = settings.get('frame_resize_width', 480)
FLOW_ANALYSIS_SCALE = float(settings.get('flow_analysis_scale', 0.75))
FLOW_RELIABLE_ERROR_THRESHOLD_PX = float(settings.get('flow_reliable_error_threshold_px', 1.0))
FLOW_TENSOR_WINDOW_SEC = float(settings.get('flow_tensor_window_sec', 1.0))
FLOW_TENSOR_HOP_SEC = float(settings.get('flow_tensor_hop_sec', 0.2))
FLOW_TENSOR_SCORE_AGGREGATION = str(settings.get('flow_tensor_score_aggregation', 'max'))
MODEL_INPUT_DTYPE = np.dtype(settings.get('model_input_dtype', 'float32')).type
FLOW_CACHE_VERSION = str(settings.get('flow_cache_version', 'sample_flow_cache_v5'))
FLOW_CACHE_METHOD = 'farneback_grid_frame_bin_apportioned_0p1s'
artifact_flow_cache_settings = settings.get('flow_cache_settings', {})
artifact_flow_method = str(artifact_flow_cache_settings.get('flow_method', FLOW_CACHE_METHOD)) if isinstance(artifact_flow_cache_settings, dict) else FLOW_CACHE_METHOD
if FLOW_CACHE_VERSION in {'sample_flow_cache_v1', 'sample_flow_cache_v2', 'sample_flow_cache_v3', 'sample_flow_cache_v4'} or artifact_flow_method != FLOW_CACHE_METHOD:
    raise ValueError('This artifact was trained with an old optical-flow cache. Re-run the training notebook.')

audio_model_artifact = artifact['score_models']['audio']
motion_model = artifact['flow_tensor_model']
MOTION_FEATURE_NAMES = flow_tensor.normalize_motion_feature_names(settings.get('motion_feature_names', motion_model.get('channels', ['flow_x', 'flow_y'])))
FLOW_TENSOR_BROAD_VIB_SCORE_CONFIG = settings.get('flow_tensor_broad_vib_score_config', {})
if len(MOTION_FEATURE_NAMES) != int(np.asarray(motion_model.get('mean_tensor')).shape[-1]):
    raise ValueError('Motion feature count does not match the trained tensor model. Re-run the training notebook.')
SYNC_SCORE_CONFIG = artifact.get('sync_score_config', {'audio_score_column': 'audio_anomaly_score', 'motion_score_column': 'motion_anomaly_score', 'max_lag_windows': 2})
FINAL_SCORE_WEIGHTS = artifact.get('final_score_weights', {'audio_anomaly_score': 0.45, 'motion_anomaly_score': 0.35, 'sync_score': 0.20})
PLOT_SCORE_COLUMNS = artifact.get('plot_score_columns', ['anomaly_score', 'audio_anomaly_score', 'motion_anomaly_score', 'sync_score'])
TRAIN_SAMPLE_IDS = set(map(str, artifact.get('train_sample_ids', [])))

INFERENCE_TARGET_SAMPLE_LIST_PATH = PROJECT_ROOT / 'data' / 'inference_target_sample_list.csv'
INFERENCE_TARGET_SAMPLE_LIST_LEGACY_PATH = PROJECT_ROOT / 'data' / 'inference_target_sample_list'
if not INFERENCE_TARGET_SAMPLE_LIST_PATH.exists() and INFERENCE_TARGET_SAMPLE_LIST_LEGACY_PATH.exists():
    INFERENCE_TARGET_SAMPLE_LIST_PATH = INFERENCE_TARGET_SAMPLE_LIST_LEGACY_PATH
TARGET_SELECTION_MODE = 'sample_list' if INFERENCE_TARGET_SAMPLE_LIST_PATH.exists() else 'random_untrained'
TARGET_SPECS: list[dict[str, Any]] = []
TARGET_RANDOM_COUNT = 8
TARGET_RANDOM_STATE = 42
TOP_N_ANOMALIES = 8
WRITE_TARGET_RAW_FLOW = False

print(f'model: {MODEL_PATH}')
print(f'model version: {artifact.get("model_version")}')
print(f'motion features: {MOTION_FEATURE_NAMES}')
print(f'target selection mode: {TARGET_SELECTION_MODE}')
print(f'train samples: {len(TRAIN_SAMPLE_IDS)}')


In [ ]:
MOVIE_PREPROCESS_MANIFEST_PATH = MOVIE_PREPROCESS_DIR / 'movie_preprocess_manifest.csv'
TRAIN_SAMPLE_LIST_PATH = PROJECT_ROOT / 'data' / 'train_sample_list.csv'


def infer_sample_id_from_preprocess_path(path_value: Any, suffix: str = '_front.mp4') -> str:
    name = Path(str(path_value)).name
    if name.endswith(suffix):
        return name[:-len(suffix)]
    return Path(name).stem


def build_preprocess_paths(sample_id: str, category: str, environment: str) -> dict[str, Path]:
    target_dir = MOVIE_PREPROCESS_DIR / str(category) / str(environment)
    return {
        'front_path': target_dir / f'{sample_id}_front.mp4',
        'rear_path': target_dir / f'{sample_id}_rear.mp4',
        'audio_path': target_dir / f'{sample_id}_audio.wav',
    }


def scan_movie_preprocess_inventory() -> pd.DataFrame:
    rows = []
    for front_path in sorted(MOVIE_PREPROCESS_DIR.glob('*/*/*_front.mp4')):
        sample_id = front_path.name[:-len('_front.mp4')]
        category = front_path.parent.parent.name
        environment = front_path.parent.name
        paths = build_preprocess_paths(sample_id, category, environment)
        if not paths['rear_path'].exists():
            continue
        rows.append({
            'sample_id': sample_id,
            'category': category,
            'environment': environment,
            **paths,
            'status': 'scanned_from_files',
            'audio_status': 'exists' if paths['audio_path'].exists() else 'missing',
        })
    return pd.DataFrame(rows)


def load_movie_preprocess_inventory(manifest_path: Path = MOVIE_PREPROCESS_MANIFEST_PATH) -> pd.DataFrame:
    if not manifest_path.exists():
        return scan_movie_preprocess_inventory()
    manifest_df = pd.read_csv(manifest_path)
    rows = []
    for _, row in manifest_df.iterrows():
        category = str(row.get('category', 'unknown'))
        environment = str(row.get('environment', 'unknown'))
        sample_id = str(row.get('sample_id', '')).strip()
        if not sample_id or sample_id.lower() == 'nan':
            sample_id = infer_sample_id_from_preprocess_path(row.get('front_output_path', row.get('input_video_path', '')))
        if not sample_id or sample_id.lower() == 'nan':
            continue
        paths = build_preprocess_paths(sample_id, category, environment)
        if not paths['front_path'].exists() or not paths['rear_path'].exists():
            continue
        rows.append({
            'sample_id': sample_id,
            'category': category,
            'environment': environment,
            **paths,
            'status': row.get('status', 'manifest'),
            'audio_status': row.get('audio_status', 'exists' if paths['audio_path'].exists() else 'missing'),
        })
    return pd.DataFrame(rows).drop_duplicates(['sample_id', 'category', 'environment']).reset_index(drop=True)


def load_train_sample_ids(path: Path = TRAIN_SAMPLE_LIST_PATH) -> set[str]:
    ids = set(TRAIN_SAMPLE_IDS)
    if path.exists():
        train_df = pd.read_csv(path)
        if 'sample_id' in train_df.columns:
            ids.update(train_df['sample_id'].astype(str).tolist())
    return ids


def select_random_untrained_targets(inventory_df: pd.DataFrame) -> pd.DataFrame:
    train_ids = load_train_sample_ids()
    candidates = inventory_df[~inventory_df['sample_id'].astype(str).isin(train_ids)].copy()
    if candidates.empty:
        candidates = inventory_df.copy()
    count = len(candidates) if int(TARGET_RANDOM_COUNT) < 0 else min(int(TARGET_RANDOM_COUNT), len(candidates))
    return candidates.sample(n=count, random_state=TARGET_RANDOM_STATE).sort_values(['category', 'environment', 'sample_id']).reset_index(drop=True)


def rows_from_target_specs(target_list_df: pd.DataFrame, inventory_df: pd.DataFrame) -> pd.DataFrame:
    if 'sample_id' not in target_list_df.columns:
        raise ValueError('target sample list must contain a sample_id column')
    rows = []
    for _, spec in target_list_df.iterrows():
        if {'front_path', 'rear_path'}.issubset(target_list_df.columns) and pd.notna(spec.get('front_path')) and pd.notna(spec.get('rear_path')):
            front_path = Path(str(spec.get('front_path')))
            rear_path = Path(str(spec.get('rear_path')))
            audio_value = spec.get('audio_path') if 'audio_path' in target_list_df.columns else None
            audio_path = Path(str(audio_value)) if pd.notna(audio_value) else front_path.with_name(f"{spec.get('sample_id')}_audio.wav")
            rows.append({
                'sample_id': str(spec.get('sample_id')),
                'category': str(spec.get('category', 'unknown')),
                'environment': str(spec.get('environment', 'unknown')),
                'front_path': front_path,
                'rear_path': rear_path,
                'audio_path': audio_path if audio_path.exists() else None,
                'plot_label': str(spec.get('plot_label', spec.get('sample_id'))),
            })
            continue

        part = inventory_df[inventory_df['sample_id'].astype(str).eq(str(spec['sample_id']))].copy()
        if 'category' in target_list_df.columns and pd.notna(spec.get('category')):
            part = part[part['category'].astype(str).eq(str(spec['category']))]
        if 'environment' in target_list_df.columns and pd.notna(spec.get('environment')):
            part = part[part['environment'].astype(str).eq(str(spec['environment']))]
        if part.empty:
            raise ValueError(f'inference target not found in movie_preprocess inventory: {dict(spec)}')
        row = part.iloc[0].copy()
        if 'plot_label' in target_list_df.columns and pd.notna(spec.get('plot_label')):
            row['plot_label'] = spec.get('plot_label')
        rows.append(row)
    return pd.DataFrame(rows).reset_index(drop=True)


def load_inference_target_sample_list(path: Path, inventory_df: pd.DataFrame) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'inference target sample list was not found: {path}')
    return rows_from_target_specs(pd.read_csv(path), inventory_df)


TARGET_INVENTORY_DF = load_movie_preprocess_inventory()
display(TARGET_INVENTORY_DF.groupby(['category', 'environment'], dropna=False).size().reset_index(name='count'))

if TARGET_SELECTION_MODE == 'sample_list':
    target_samples = load_inference_target_sample_list(INFERENCE_TARGET_SAMPLE_LIST_PATH, TARGET_INVENTORY_DF)
elif TARGET_SELECTION_MODE == 'manual':
    target_samples = rows_from_target_specs(pd.DataFrame(TARGET_SPECS), TARGET_INVENTORY_DF)
elif TARGET_SELECTION_MODE == 'random_untrained':
    target_samples = select_random_untrained_targets(TARGET_INVENTORY_DF)
else:
    raise ValueError("TARGET_SELECTION_MODE must be 'sample_list', 'manual', or 'random_untrained'")

if target_samples.empty:
    raise ValueError('No target samples were selected')

selected_targets_df = target_samples.copy()
display(selected_targets_df[['sample_id', 'category', 'environment', 'front_path', 'rear_path', 'audio_path']])


In [ ]:
# ------------------------------------------------------------
# セル概要: 共通特徴抽出パイプライン設定
# ------------------------------------------------------------
# - 学習artifactの設定から、推論用のflow cache・motion tensor・音声特徴抽出設定を作ります。
# - cache利用と音声/動きスコア結合は src/forklift_features/pipeline.py に集約しています。
# ------------------------------------------------------------

FLOW_CACHE_SETTINGS = feature_pipeline.build_flow_cache_settings(
    cache_version=FLOW_CACHE_VERSION,
    use_front=USE_FRONT,
    use_rear=USE_REAR,
    flow_sample_sec=FLOW_SAMPLE_SEC,
    frame_resize_width=FRAME_RESIZE_WIDTH,
    flow_analysis_scale=FLOW_ANALYSIS_SCALE,
    flow_grid=FLOW_GRID,
    flow_reliable_error_threshold_px=FLOW_RELIABLE_ERROR_THRESHOLD_PX,
    flow_method=FLOW_CACHE_METHOD,
)
RAW_FLOW_PIPELINE_KWARGS = {
    'flow_cache_settings': FLOW_CACHE_SETTINGS,
    'cache_dir': FEATURE_CACHE_DIR,
    'use_front': USE_FRONT,
    'use_rear': USE_REAR,
    'flow_sample_sec': FLOW_SAMPLE_SEC,
    'frame_resize_width': FRAME_RESIZE_WIDTH,
    'flow_analysis_scale': FLOW_ANALYSIS_SCALE,
    'flow_grid': FLOW_GRID,
    'flow_reliable_error_threshold_px': FLOW_RELIABLE_ERROR_THRESHOLD_PX,
    'enable_cache': True,
}
AUDIO_PIPELINE_KWARGS = {
    'audio_sr': AUDIO_SR,
    'window_sec': WINDOW_SEC,
    'hop_sec': FLOW_TENSOR_HOP_SEC,
    'n_mels': N_MELS,
    'label_default': None,
}
MOTION_CAMERAS = feature_pipeline.enabled_cameras(use_front=USE_FRONT, use_rear=USE_REAR)


In [ ]:
target_results: dict[str, dict[str, Any]] = {}
all_feature_parts: list[pd.DataFrame] = []
all_raw_flow_parts: list[pd.DataFrame] = []
all_scored_parts: list[pd.DataFrame] = []
cache_rows: list[dict[str, Any]] = []

for record in tqdm(target_samples.to_dict('records'), total=len(target_samples), desc='score targets'):
    sample = pd.Series(record)
    sample_id = str(sample.get('sample_id', 'unknown'))
    sample_output_dir = TARGET_OUTPUT_ROOT / safe_path_part(sample_id)
    sample_output_dir.mkdir(parents=True, exist_ok=True)

    raw_flow_df, cache_info = feature_pipeline.load_or_extract_raw_flow(sample, **RAW_FLOW_PIPELINE_KWARGS)
    cache_rows.append(cache_info)
    if len(raw_flow_df):
        all_raw_flow_parts.append(raw_flow_df)
        raw_flow_df.to_csv(sample_output_dir / f'{safe_path_part(sample_id)}_raw_flow_0p1s.csv', index=False)

    X_target, meta_df = flow_tensor.build_flow_tensor_windows(
        raw_flow_df,
        cameras=MOTION_CAMERAS,
        flow_sample_sec=FLOW_SAMPLE_SEC,
        window_sec=FLOW_TENSOR_WINDOW_SEC,
        hop_sec=FLOW_TENSOR_HOP_SEC,
        grid=FLOW_GRID,
        feature_names=MOTION_FEATURE_NAMES,
        broad_vib_score_config=FLOW_TENSOR_BROAD_VIB_SCORE_CONFIG,
    )
    if len(X_target):
        X_target_model = X_target.astype(MODEL_INPUT_DTYPE, copy=False)
        motion_raw, motion_scores = flow_tensor.score_flow_tensor_windows(X_target_model, motion_model)
        motion_window_scores = meta_df.copy()
        motion_window_scores['category'] = sample.get('category', 'unknown')
        motion_window_scores['environment'] = sample.get('environment', 'unknown')
        motion_window_scores['motion_anomaly_score_raw'] = motion_raw
        motion_window_scores['motion_anomaly_score'] = motion_scores
        motion_explanation_df = flow_tensor.explain_flow_tensor_windows(
            X_target_model,
            motion_model,
            meta_df,
        )
        if len(motion_explanation_df):
            for col in motion_explanation_df.columns:
                motion_window_scores[col] = motion_explanation_df[col].to_numpy()
        motion_scores_df = flow_tensor.aggregate_camera_scores(
            motion_window_scores,
            method=FLOW_TENSOR_SCORE_AGGREGATION,
            score_col='motion_anomaly_score',
            raw_score_col='motion_anomaly_score_raw',
        )
    else:
        motion_window_scores = pd.DataFrame()
        motion_scores_df = pd.DataFrame()

    audio_features_df = feature_pipeline.build_audio_features(sample, **AUDIO_PIPELINE_KWARGS)
    if len(audio_features_df):
        audio_raw, audio_scores = scoring.score_isolation_forest_artifact(audio_features_df, audio_model_artifact)
        audio_scores_df = audio_features_df.copy()
        audio_scores_df['audio_anomaly_score_raw'] = audio_raw
        audio_scores_df['audio_anomaly_score'] = audio_scores
    else:
        audio_scores_df = pd.DataFrame()

    scored_df = feature_pipeline.combine_audio_motion_scores(audio_scores_df, motion_scores_df, hop_sec=FLOW_TENSOR_HOP_SEC)
    scored_df = scoring.add_composite_scores(scored_df, sync_score_config=SYNC_SCORE_CONFIG, final_score_weights=FINAL_SCORE_WEIGHTS)
    if len(scored_df):
        scored_df.to_csv(sample_output_dir / f'{safe_path_part(sample_id)}_window_scores.csv', index=False)
        scored_df.nlargest(TOP_N_ANOMALIES, 'anomaly_score').to_csv(sample_output_dir / f'{safe_path_part(sample_id)}_top_windows.csv', index=False)
        all_scored_parts.append(scored_df)
    if len(audio_features_df):
        audio_features_df.to_csv(sample_output_dir / f'{safe_path_part(sample_id)}_features.csv', index=False)
        all_feature_parts.append(audio_features_df)

    target_results[sample_id] = {
        'sample': sample.to_dict(),
        'raw_flow_df': raw_flow_df,
        'audio_features_df': audio_features_df,
        'motion_window_scores_df': motion_window_scores,
        'scored_df': scored_df,
    }

all_target_features_df = pd.concat(all_feature_parts, ignore_index=True) if all_feature_parts else pd.DataFrame()
all_target_raw_flow_df = pd.concat(all_raw_flow_parts, ignore_index=True) if all_raw_flow_parts else pd.DataFrame()
all_target_scored_df = pd.concat(all_scored_parts, ignore_index=True) if all_scored_parts else pd.DataFrame()
cache_manifest_df = pd.DataFrame(cache_rows)

all_target_features_df.to_csv(TARGET_COMPARISON_DIR / 'all_target_features.csv', index=False)
all_target_raw_flow_df.to_csv(TARGET_COMPARISON_DIR / 'all_target_raw_flow_0p1s.csv', index=False)
all_target_scored_df.to_csv(TARGET_COMPARISON_DIR / 'all_target_window_scores.csv', index=False)
cache_manifest_df.to_csv(TARGET_COMPARISON_DIR / 'flow_cache_manifest.csv', index=False)

motion_attribution_cols = [
    'motion_top_camera',
    'motion_top_channel',
    'motion_top_grid_row',
    'motion_top_grid_col',
    'motion_top_grid_label',
    'motion_top_grid_channel',
    'motion_flow_x_contribution',
    'motion_flow_y_contribution',
    'motion_t_flow_x_contribution',
    'motion_t_flow_y_contribution',
    'motion_flow_x_broad_vib_score_contribution',
    'motion_flow_y_broad_vib_score_contribution',
    'motion_t_flow_x_broad_vib_score_contribution',
    'motion_t_flow_y_broad_vib_score_contribution',
    'motion_top_grid_contribution',
]
motion_attribution_display_cols = [
    'motion_top_camera',
    'motion_top_grid_channel',
    'motion_top_grid_label',
]

if len(all_target_raw_flow_df):
    raw_flow_display_cols = [c for c in ['video_id', 'target_label', 'time', 'front_flow_x_mean', 'front_flow_y_mean', 'rear_flow_x_mean', 'rear_flow_y_mean'] if c in all_target_raw_flow_df.columns]
    if raw_flow_display_cols:
        display(all_target_raw_flow_df[raw_flow_display_cols].head(12))
if len(all_target_scored_df):
    score_display_cols = [c for c in [
        'video_id', 'target_label', 'time', 'anomaly_score',
        'audio_anomaly_score', 'motion_anomaly_score', 'sync_score', 'anomaly_score_smooth',
        *motion_attribution_display_cols,
    ] if c in all_target_scored_df.columns]
    display(all_target_scored_df[score_display_cols].head(12))

    scored_for_summary = all_target_scored_df.copy()
    if 'window_start_sec' not in scored_for_summary.columns:
        scored_for_summary['window_start_sec'] = np.nan
    summary_time = pd.to_numeric(scored_for_summary['time'], errors='coerce').fillna(0.0) if 'time' in scored_for_summary.columns else pd.Series(0.0, index=scored_for_summary.index)
    fallback_start_sec = summary_time - 0.5 * float(FLOW_TENSOR_WINDOW_SEC)
    scored_for_summary['window_start_sec'] = pd.to_numeric(scored_for_summary['window_start_sec'], errors='coerce').fillna(fallback_start_sec).clip(lower=0.0)
    if 'window_end_sec' not in scored_for_summary.columns:
        scored_for_summary['window_end_sec'] = scored_for_summary['window_start_sec'] + float(FLOW_TENSOR_WINDOW_SEC)
    if 'window_center_sec' not in scored_for_summary.columns:
        scored_for_summary['window_center_sec'] = 0.5 * (scored_for_summary['window_start_sec'] + pd.to_numeric(scored_for_summary['window_end_sec'], errors='coerce').fillna(scored_for_summary['window_start_sec']))

    best_window_indices = scored_for_summary.groupby('video_id')['anomaly_score'].idxmax()
    best_windows_df = scored_for_summary.loc[best_window_indices].copy()
    best_windows_df = best_windows_df.rename(columns={
        'window_start_sec': 'max_anomaly_window_start_sec',
        'window_end_sec': 'max_anomaly_window_end_sec',
        'window_center_sec': 'max_anomaly_window_center_sec',
        'anomaly_score': 'max_anomaly_score',
        'audio_anomaly_score': 'audio_anomaly_score_at_max_anomaly',
        'motion_anomaly_score': 'motion_anomaly_score_at_max_anomaly',
        'sync_score': 'sync_score_at_max_anomaly',
        'final_anomaly_score': 'final_anomaly_score_at_max_anomaly',
    })
    summary_extra_df = (
        scored_for_summary
        .groupby('video_id', as_index=False)
        .agg(
            top5_mean_anomaly_score=('anomaly_score', scoring.topk_mean),
            p95_anomaly_score=('anomaly_score', lambda s: float(s.quantile(0.95))),
            n_windows=('anomaly_score', 'size'),
        )
    )
    summary_base_cols = [
        'video_id', 'target_category', 'target_environment', 'target_label',
        'max_anomaly_window_start_sec', 'max_anomaly_window_end_sec', 'max_anomaly_window_center_sec',
        'max_anomaly_score', 'sync_score_at_max_anomaly',
        'audio_anomaly_score_at_max_anomaly', 'motion_anomaly_score_at_max_anomaly',
        'final_anomaly_score_at_max_anomaly',
    ]
    target_video_summary = (
        best_windows_df[[c for c in [*summary_base_cols, *motion_attribution_cols] if c in best_windows_df.columns]]
        .merge(summary_extra_df, on='video_id', how='left')
        .sort_values('max_anomaly_score', ascending=False)
        .reset_index(drop=True)
    )
    inference_base_cols = [
        'video_id', 'target_category', 'target_environment', 'max_anomaly_window_start_sec',
        'max_anomaly_score', 'sync_score_at_max_anomaly',
        'audio_anomaly_score_at_max_anomaly', 'motion_anomaly_score_at_max_anomaly',
    ]
    inference_result_table = (
        target_video_summary[[c for c in [*inference_base_cols, *motion_attribution_display_cols] if c in target_video_summary.columns]]
        .rename(columns={
            'video_id': 'ファイル名',
            'target_category': '正解ラベル',
            'target_environment': '環境',
            'max_anomaly_window_start_sec': '最大異常窓開始時刻',
            'max_anomaly_score': '異常スコア',
            'sync_score_at_max_anomaly': '同期異常スコア',
            'audio_anomaly_score_at_max_anomaly': '音声異常スコア',
            'motion_anomaly_score_at_max_anomaly': '動き異常スコア',
            'motion_top_camera': '動き異常カメラ',
            'motion_top_grid_channel': '動き異常成分',
            'motion_top_grid_label': '動き異常グリッド',
        })
        .sort_values('異常スコア', ascending=False)
        .reset_index(drop=True)
    )
    window_metric_cols = [
        'video_id', 'target_category', 'target_environment', 'target_label',
        'window_start_sec', 'time', *PLOT_SCORE_COLUMNS, 'final_anomaly_score',
        'anomaly_score_smooth', *motion_attribution_cols,
    ]
    target_top_windows = pd.concat(
        [g.nlargest(TOP_N_ANOMALIES, 'anomaly_score') for _, g in scored_for_summary.groupby('video_id', sort=False)],
        ignore_index=True,
    )[[c for c in window_metric_cols if c in scored_for_summary.columns]]
    all_target_key_metrics_df = scored_for_summary[[c for c in window_metric_cols if c in scored_for_summary.columns]].copy()

    target_video_summary.to_csv(TARGET_COMPARISON_DIR / 'target_video_scores.csv', index=False)
    inference_result_table.to_csv(TARGET_COMPARISON_DIR / 'target_inference_result_table.csv', index=False)
    target_top_windows.to_csv(TARGET_COMPARISON_DIR / 'target_top_windows.csv', index=False)
    all_target_key_metrics_df.to_csv(TARGET_COMPARISON_DIR / 'all_target_key_metrics.csv', index=False)
    for sample_id, result in target_results.items():
        scored_df = result['scored_df']
        if scored_df.empty:
            continue
        sample_output_dir = TARGET_OUTPUT_ROOT / safe_path_part(sample_id)
        target_video_summary[target_video_summary['video_id'].astype(str).eq(str(sample_id))].to_csv(sample_output_dir / f'{safe_path_part(sample_id)}_video_score.csv', index=False)

    display(inference_result_table)
    display(target_video_summary)
    display(target_top_windows)
    display(all_target_key_metrics_df.head(12))
else:
    target_video_summary = pd.DataFrame()
    inference_result_table = pd.DataFrame()
    target_top_windows = pd.DataFrame()
    all_target_key_metrics_df = pd.DataFrame()
    print('No scored windows were produced.')

print('Graph output skipped; inference results are displayed as tables above.')
print(f'saved comparison outputs: {TARGET_COMPARISON_DIR}')
